In [2]:
import pandas as pd

file_path = "../data/UserBehavior.csv"

columns = [
    "user_id",
    "item_id",
    "category_id",
    "behavior_type",
    "timestamp"
]

df = pd.read_csv(
    file_path,
    names=columns,
    nrows=5
)

df

,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [2]:
df["datetime"] = pd.to_datetime(
    df["timestamp"],
    unit="s"
)

df

,user_id,item_id,category_id,behavior_type,timestamp,datetime
0,1,2268318,2520377,pv,1511544070,2017-11-24 17:21:10
1,1,2333346,2520771,pv,1511561733,2017-11-24 22:15:33
2,1,2576651,149192,pv,1511572885,2017-11-25 01:21:25
3,1,3830808,4181361,pv,1511593493,2017-11-25 07:04:53
4,1,4365585,2520377,pv,1511596146,2017-11-25 07:49:06


In [3]:
sample = pd.read_csv(
    file_path,
    names=columns,
    nrows=10000
)

sample["behavior_type"].value_counts()

behavior_type
pv      8986
cart     537
fav      298
buy      179
Name: count, dtype: int64

In [4]:
sample["datetime"] = (
    pd.to_datetime(sample["timestamp"], unit="s", utc=True)
    .dt.tz_convert("Asia/Shanghai")
)

sample[["timestamp", "datetime"]].head()

,timestamp,datetime
0,1511544070,2017-11-25 01:21:10+08:00
1,1511561733,2017-11-25 06:15:33+08:00
2,1511572885,2017-11-25 09:21:25+08:00
3,1511593493,2017-11-25 15:04:53+08:00
4,1511596146,2017-11-25 15:49:06+08:00


In [5]:
##检查缺失值
sample.isna().sum()

user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
datetime         0
dtype: int64

In [6]:
# 检查样本中是否存在完全重复的记录
sample.duplicated().sum()

np.int64(0)

In [7]:
# 查看样本中每一列的数据类型
sample.dtypes

user_id                                  int64
item_id                                  int64
category_id                              int64
behavior_type                           object
timestamp                                int64
datetime         datetime64[ns, Asia/Shanghai]
dtype: object

In [8]:
# 扫描完整 CSV：
# 统计总记录数，并检查整个数据集中每一列的缺失值数量

total_rows = 0
missing_counts = pd.Series(0, index=columns)

for chunk in pd.read_csv(
    file_path,
    names=columns,
    chunksize=1_000_000
):
    total_rows += len(chunk)
    missing_counts += chunk.isna().sum()

print("总记录数：", total_rows)
print("\n各列缺失值数量：")
print(missing_counts)

总记录数： 100150807

各列缺失值数量：
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64


In [3]:
# 扫描完整数据集，找出最早和最晚的时间戳
# 目的：确认这 1 亿条行为记录实际覆盖的时间范围

min_timestamp = None
max_timestamp = None

for chunk in pd.read_csv(
    file_path,
    names=columns,
    chunksize=1_000_000
):
    chunk_min = chunk["timestamp"].min()
    chunk_max = chunk["timestamp"].max()

    if min_timestamp is None or chunk_min < min_timestamp:
        min_timestamp = chunk_min

    if max_timestamp is None or chunk_max > max_timestamp:
        max_timestamp = chunk_max

# 将 Unix 时间戳转换成北京时间，方便阅读
min_datetime = (
    pd.to_datetime(min_timestamp, unit="s", utc=True)
    .tz_convert("Asia/Shanghai")
)

max_datetime = (
    pd.to_datetime(max_timestamp, unit="s", utc=True)
    .tz_convert("Asia/Shanghai")
)

print("最早时间：", min_datetime)
print("最晚时间：", max_datetime)

最早时间： 1902-05-08 06:32:46+08:00
最晚时间： 2037-04-09 13:22:35+08:00


In [4]:
# 统计正常分析时间范围内和范围外的记录数量
# 目的：判断异常时间记录有多少，再决定是否过滤

start_time = int(
    pd.Timestamp("2017-11-25 00:00:00", tz="Asia/Shanghai").timestamp()
)

end_time = int(
    pd.Timestamp("2017-12-04 00:00:00", tz="Asia/Shanghai").timestamp()
)

normal_rows = 0
abnormal_rows = 0

for chunk in pd.read_csv(
    file_path,
    names=columns,
    chunksize=1_000_000
):
    # 正常记录：时间在 2017-11-25 00:00:00 至 2017-12-04 00:00:00 之间
    normal_mask = (
        (chunk["timestamp"] >= start_time) &
        (chunk["timestamp"] < end_time)
    )

    normal_rows += normal_mask.sum()
    abnormal_rows += (~normal_mask).sum()

print("正常时间范围记录数：", normal_rows)
print("异常时间记录数：", abnormal_rows)
print("异常记录占比：", abnormal_rows / (normal_rows + abnormal_rows))

正常时间范围记录数： 100095231
异常时间记录数： 55576
异常记录占比： 0.0005549231370646869


In [5]:
# 生成清洗后的数据文件
# 目的：过滤异常时间记录，并保留正常分析区间内的数据

output_file = "../data/UserBehavior_clean.csv"

first_chunk = True

for chunk in pd.read_csv(
    file_path,
    names=columns,
    chunksize=1_000_000
):
    # 只保留正常时间范围内的数据
    clean_chunk = chunk[
        (chunk["timestamp"] >= start_time) &
        (chunk["timestamp"] < end_time)
    ].copy()

    # 把 Unix 时间戳转换为北京时间
    clean_chunk["datetime"] = (
        pd.to_datetime(clean_chunk["timestamp"], unit="s", utc=True)
        .dt.tz_convert("Asia/Shanghai")
        .dt.tz_localize(None)
    )

    # 分块写入清洗后的 CSV
    clean_chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("清洗后的文件已生成：", output_file)

清洗后的文件已生成： ../data/UserBehavior_clean.csv


In [6]:
# 统计清洗后数据中的独立用户数量
# 目的：了解用户规模，为后面的“按用户抽样”确定合适的样本量

unique_users = set()

for chunk in pd.read_csv(
    "../data/UserBehavior_clean.csv",
    usecols=["user_id"],
    chunksize=1_000_000
):
    # 将这一批数据里的用户ID加入集合
    # set 会自动去掉重复的 user_id
    unique_users.update(chunk["user_id"].unique())

print("独立用户数量：", len(unique_users))

独立用户数量： 987991


In [7]:
# 从全部用户中随机抽取 50,000 个用户
# 目的：缩小本地分析数据量，同时保留被抽中用户的完整行为记录

import random

random.seed(42)

sampled_users = set(
    random.sample(list(unique_users), 50_000)
)

print("抽样用户数量：", len(sampled_users))

抽样用户数量： 50000


In [8]:
# 提取抽样用户的全部行为记录
# 目的：保留这 5 万个用户完整的浏览、收藏、加购、购买行为链，
#      并生成后续 SQLite / SQL 分析使用的数据文件

sample_output_file = "../data/UserBehavior_sample.csv"

first_chunk = True
sample_rows = 0

for chunk in pd.read_csv(
    "../data/UserBehavior_clean.csv",
    chunksize=1_000_000
):
    # 只保留 user_id 属于 sampled_users 的记录
    sampled_chunk = chunk[
        chunk["user_id"].isin(sampled_users)
    ]

    # 累计抽样后的记录数
    sample_rows += len(sampled_chunk)

    # 分批写入新的 CSV 文件
    sampled_chunk.to_csv(
        sample_output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("抽样用户数量：", len(sampled_users))
print("抽样后行为记录数：", sample_rows)
print("抽样文件已生成：", sample_output_file)

抽样用户数量： 50000
抽样后行为记录数： 5043033
抽样文件已生成： ../data/UserBehavior_sample.csv


In [9]:
# 创建 SQLite 数据库，并将抽样后的用户行为数据导入数据库
# 目的：建立后续 SQL 商业分析使用的 user_behavior 数据表

import sqlite3

# 创建或连接 SQLite 数据库
conn = sqlite3.connect("../data/taobao_analysis.db")

# 分块读取 CSV，避免一次性占用过多内存
for chunk in pd.read_csv(
    "../data/UserBehavior_sample.csv",
    chunksize=500_000
):
    # 将每一批数据追加写入数据库中的 user_behavior 表
    chunk.to_sql(
        "user_behavior",
        conn,
        if_exists="append",
        index=False
    )

print("数据已成功导入 SQLite。")

数据已成功导入 SQLite。
